In [ ]:
!pip install -q chromadb

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 52.0/52.0 kB 1.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.3/23.3 MB 68.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 278.2/278.2 kB 22.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.7/4.7 MB 103.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.2/18.2 MB 81.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.8/71.8 kB 5.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 170.9/170.9 kB 13.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.3/61.3 kB 4.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 203.7/203.7 kB 16.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 71.6/71.6 kB 5.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.6/60.6 kB 4.8 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the 

In [ ]:
import chromadb
print(chromadb.__version__)

1.5.9


In [ ]:
from google.colab import files
files.upload()

Saving playbooks_fixed.json to playbooks_fixed.json


{'playbooks_fixed.json': b'[\n  {\n    "title": "Pod OOMKilled: Insufficient Memory Limit",\n    "category": "Memory",\n    "source": "Kubernetes",\n    "symptoms": "Pods frequently restart with `OOMKilled` status. `kubectl describe pod <pod-name>` shows `Last State: Terminated` and `Reason: OOMKilled`. Application logs might be cut short or indicate abrupt termination.",\n    "root_cause": "The application running inside the container attempts to allocate more memory than specified by its `resources.limits.memory` in the pod\'s container definition. The kernel\'s OOM killer terminates the process.",\n    "resolution_steps": [\n      "Verify the `OOMKilled` status using `kubectl get pod <pod-name> -o yaml`.",\n      "Check container memory usage metrics (e.g., Prometheus/Grafana) to identify peak memory consumption.",\n      "Analyze application logs for clues about memory spikes or growth patterns.",\n      "Increase the `resources.limits.memory` for the affected container in its Depl

In [ ]:
import json

# Load the cleaned playbooks
with open("playbooks_fixed.json", encoding="utf-8") as f:
    playbooks = json.load(f)
print(f"Loaded {len(playbooks)} playbooks")

# Create a PERSISTENT Chroma client (saved to disk, survives restarts)
client = chromadb.PersistentClient(path="./chroma_playbooks")

# Create (or reset) the collection. cosine distance is best for text similarity.
try:
    client.delete_collection("playbooks")   # start clean if re-running
except Exception:
    pass
collection = client.create_collection(
    name="playbooks",
    metadata={"hnsw:space": "cosine"},
)

# ---- Build the search text + metadata for each playbook ----
# Search text = what we match against (symptoms + root cause + title).
# Full details are stored in metadata so we can retrieve them after a match.
documents = []   # the text we embed & search
metadatas = []   # full playbook details
ids = []

for i, pb in enumerate(playbooks):
    # the text used for semantic matching
    search_text = (
        f"Title: {pb.get('title','')}\n"
        f"Category: {pb.get('category','')} | Source: {pb.get('source','')}\n"
        f"Symptoms: {pb.get('symptoms','')}\n"
        f"Root cause: {pb.get('root_cause','')}"
    )
    documents.append(search_text)

    # store everything as metadata (Chroma metadata must be str/int/float/bool)
    metadatas.append({
        "title": pb.get("title", ""),
        "category": pb.get("category", ""),
        "source": pb.get("source", ""),
        "severity_typical": pb.get("severity_typical", ""),
        "resolution_steps": json.dumps(pb.get("resolution_steps", []), ensure_ascii=False),
        "code_fix": pb.get("code_fix", ""),
        "internal_note": pb.get("internal_note", ""),
    })
    ids.append(f"pb_{i}")

# Add everything to Chroma — it embeds & indexes automatically
collection.add(documents=documents, metadatas=metadatas, ids=ids)
print(f"✅ Indexed {collection.count()} playbooks into Chroma")

Loaded 102 playbooks


/root/.cache/chroma/onnx_models/all-MiniLM-L6-v2/onnx.tar.gz: 100%|██████████| 79.3M/79.3M [00:01<00:00, 43.7MiB/s]


✅ Indexed 102 playbooks into Chroma


In [ ]:
# collection.query(
#     query_texts=[
#         "pods restarting because memory usage spikes"
#     ],
#     n_results=3
# )

In [ ]:
import shutil
shutil.make_archive("chroma_playbooks", "zip", "chroma_playbooks")
print("Done")

from google.colab import files
files.download("chroma_playbooks.zip")

Done


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>